<a href="https://colab.research.google.com/github/DikozWorld/Chisl-metods-labs/blob/main/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# ЗАДАНИЕ 1: ОДУ 1-го порядка (Вариант 8)

def f1(x, y):
    # уравнение: y' = x*e^(-x^2) - 2xy
    return x * np.exp(-x**2) - 2 * x * y

def exact_sol_1(x):
    # точное решение: φ(x) = 0.5 * x^2 * e^(-x^2)
    return 0.5 * (x**2) * np.exp(-x**2)

a1 = 0.0      # Начало
b1 = 1.0      # Конец
y0_1 = 0.0    # Начальное условие y(0) = 0

def euler_method(f, a, b, y0, h):
    x = np.arange(a, b + h/2, h)
    y = np.zeros(len(x))
    y[0] = y0
    for i in range(len(x) - 1):
        y[i+1] = y[i] + h * f(x[i], y[i])
    return x, y

def euler_cauchy_method(f, a, b, y0, h):
    x = np.arange(a, b + h/2, h)
    y = np.zeros(len(x))
    y[0] = y0
    for i in range(len(x) - 1):
        y_pred = y[i] + h * f(x[i], y[i])
        y[i+1] = y[i] + h/2 * (f(x[i], y[i]) + f(x[i+1], y_pred))
    return x, y

def rk4_method(f, a, b, y0, h):
    x = np.arange(a, b + h/2, h)
    y = np.zeros(len(x))
    y[0] = y0
    for i in range(len(x) - 1):
        k1 = h * f(x[i], y[i])
        k2 = h * f(x[i] + h/2, y[i] + k1/2)
        k3 = h * f(x[i] + h/2, y[i] + k2/2)
        k4 = h * f(x[i] + h, y[i] + k3)
        y[i+1] = y[i] + (k1 + 2*k2 + 2*k3 + k4) / 6
    return x, y

def run_task_1(h):
    x, y_euler = euler_method(f1, a1, b1, y0_1, h)
    _, y_ec = euler_cauchy_method(f1, a1, b1, y0_1, h)
    _, y_rk4 = rk4_method(f1, a1, b1, y0_1, h)
    y_exact = exact_sol_1(x)

    df = pd.DataFrame({
        'x': x,
        'Точное y': y_exact,
        'Эйлер': y_euler,
        'Погр. Эйлера': np.abs(y_exact - y_euler),
        'Эйлер-Коши': y_ec,
        'Погр. Э-К': np.abs(y_exact - y_ec),
        'Рунге-Кутт': y_rk4,
        'Погр. Р-К': np.abs(y_exact - y_rk4)
    })
    print(f"\nРезультаты Задания 1 (Шаг h = {h})")
    display(df)

# ЗАДАНИЕ 2: Система ОДУ 1-го порядка из 2-го порядка (Вариант 8)

def f2(x, y1, y2):
    # система: y2' = 3x + y2/x - y1/x^2
    return 3 * x + y2 / x - y1 / (x**2)

def exact_sol_2(x):
    # Точное решение: φ(x) = 0.75 * x^3
    return 0.75 * (x**3)

a2 = 1.0       # Начало
b2 = 2.0       # Конец
y1_0 = 0.75    # Начальное условие y(1)
y2_0 = 2.25    # Начальное условие y'(1)

def rk4_system(f_sys, a, b, y1_start, y2_start, h):
    x = np.arange(a, b + h/2, h)
    y1 = np.zeros(len(x))
    y2 = np.zeros(len(x))
    y1[0] = y1_start
    y2[0] = y2_start

    for i in range(len(x) - 1):
        xi, y1i, y2i = x[i], y1[i], y2[i]

        # Первое уравнение: y1' = y2
        k1_1 = h * y2i
        k1_2 = h * f_sys(xi, y1i, y2i)

        k2_1 = h * (y2i + k1_2/2)
        k2_2 = h * f_sys(xi + h/2, y1i + k1_1/2, y2i + k1_2/2)

        k3_1 = h * (y2i + k2_2/2)
        k3_2 = h * f_sys(xi + h/2, y1i + k2_1/2, y2i + k2_2/2)

        k4_1 = h * (y2i + k3_2)
        k4_2 = h * f_sys(xi + h, y1i + k3_1, y2i + k3_2)

        y1[i+1] = y1i + (k1_1 + 2*k2_1 + 2*k3_1 + k4_1) / 6
        y2[i+1] = y2i + (k1_2 + 2*k2_2 + 2*k3_2 + k4_2) / 6

    return x, y1

def run_task_2():
    h = 0.2
    x_h, y1_h = rk4_system(f2, a2, b2, y1_0, y2_0, h)
    x_h2, y1_h2 = rk4_system(f2, a2, b2, y1_0, y2_0, h/2)

    # Берем каждый второй узел из сетки с половинным шагом для сравнения
    y1_h2_filtered = y1_h2[::2]
    y_exact = exact_sol_2(x_h)

    df = pd.DataFrame({
        'x': x_h,
        'Точное y(x)': y_exact,
        f'Р-К (h={h})': y1_h,
        f'Погр. (h={h})': np.abs(y_exact - y1_h),
        f'Р-К (h={h/2})': y1_h2_filtered,
        f'Погр. (h={h/2})': np.abs(y_exact - y1_h2_filtered)
    })
    print("\nРезультаты Задания 2 (Система ОДУ)")
    display(df)

run_task_1(0.2)
run_task_1(0.1)
run_task_2()


Результаты Задания 1 (Шаг h = 0.2)


,x,Точное y,Эйлер,Погр. Эйлера,Эйлер-Коши,Погр. Э-К,Рунге-Кутт,Погр. Р-К
0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00
1,0.2,0.019216,0.000000,0.019216,0.019216,3.469447e-18,0.019215,6.010731e-07
2,0.4,0.068172,0.038432,0.029740,0.067260,9.116138e-04,0.068169,2.220472e-06
3,0.6,0.125582,0.100454,0.025128,0.122865,2.716687e-03,0.125576,6.059678e-06
4,0.8,0.168734,0.160066,0.008667,0.163829,4.904132e-03,0.168718,1.523046e-05
5,1.0,0.183940,0.193212,0.009272,0.177434,6.505812e-03,0.183909,3.056801e-05



Результаты Задания 1 (Шаг h = 0.1)


,x,Точное y,Эйлер,Погр. Эйлера,Эйлер-Коши,Погр. Э-К,Рунге-Кутт,Погр. Р-К
0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00
1,0.1,0.004950,0.000000,0.004950,0.004950,8.673617e-19,0.004950,1.015731e-08
2,0.2,0.019216,0.009900,0.009315,0.019164,5.193340e-05,0.019216,3.921465e-08
3,0.3,0.041127,0.028720,0.012407,0.040969,1.578556e-04,0.041127,8.668538e-08
4,0.4,0.068172,0.054415,0.013757,0.067855,3.168369e-04,0.068171,1.569620e-07
5,0.5,0.097350,0.084148,0.013203,0.096828,5.223251e-04,0.097350,2.619441e-07
6,0.6,0.125582,0.114673,0.010909,0.124822,7.601473e-04,0.125581,4.200018e-07
7,0.7,0.150093,0.142773,0.007321,0.149085,1.008208e-03,0.150093,6.498471e-07
8,0.8,0.168734,0.165668,0.003065,0.167495,1.238438e-03,0.168733,9.602992e-07
9,0.9,0.180168,0.181345,0.001177,0.178747,1.420818e-03,0.180166,1.339342e-06



Результаты Задания 2 (Система ОДУ)


,x,Точное y(x),Р-К (h=0.2),Погр. (h=0.2),Р-К (h=0.1),Погр. (h=0.1)
0,1.0,0.750,0.750000,0.000000,0.750000,0.000000e+00
1,1.2,1.296,1.296008,0.000008,1.296001,5.730406e-07
2,1.4,2.058,2.058017,0.000017,2.058001,1.263435e-06
3,1.6,3.072,3.072028,0.000028,3.072002,2.048574e-06
4,1.8,4.374,4.374041,0.000041,4.374003,2.913752e-06
5,2.0,6.000,6.000054,0.000054,6.000004,3.848539e-06
